# Part 1:Three consensus clusters are evaluated automatically using the 22 M_signatures.

In [ ]:
library(dplyr)
library(tidyverse)
library(ConsensusClusterPlus)
library(caret)
library(ggplot2)
library(clusterProfiler)
library(reshape2)
library(plyr)
library(pheatmap)
library(patchwork)
library(Seurat)
library(paletteer)
library(tidyverse)
library(ggpubr) 
library(ggsci)
library(cluster)
library(FCPS)
library(clustree)
library(ComplexHeatmap)
library(circlize)
source("./Function/3.2.k_means.R") 
source("./Function/3.1.hclust.R") 
source("./Function/3.3.ConstructCAM3.0.R") 
source("./Function/3.4.Num_Clusters.R")
source("./Function/3.4.2.PAC.R")
source("./Function/3.4.3.Silhouette.R")
source("./Function/3.4.1.CopheneticCorCoef.R")
source("./Function/3.4.4.AnalyzeICL.R")


input:
> (1) seuratObj: Seurat object storing processed single-cell expression profiles
> 
> (2) signatureList: List of signature sets for base clustering

In [ ]:
# 1. Base Clustering
####
# Extract expression matrix
erMatrix<-as.data.frame(seuratObj@assays$SCT@data)
# Perform base clustering to identify the optimal clustering results for both
results.CAM <- lapply(signatureList, function(x){
  ErMatrix <- erMatrix[unlist(x),]%>%na.omit()
  kmean.CAM<-k_means(ErMatrix,seed = 1234,nc.min = 2,nc.max = 8)
  hclust.CAM<-hclustfunc(ErMatrix,nc.min = 2,nc.max = 8,seed = 1234,nc.method = "ward.D2")
  results.CAM<-append(kmean.CAM,hclust.CAM) 
})

In [ ]:
# 2 Combine Base Clustering Results and Construct Consensus Matrix
####
Results.CAM <- lapply(results.CAM, function(x){
  CAM1<-as.data.frame(ConstructCAM(x$kmeans_clus))
  CAM2<-as.data.frame(ConstructCAM(x$hclust_clus))
  CAM<-(CAM1+CAM2)/2 
})
CAM <- matrix(0, ncol(Results.CAM[[1]]), ncol(Results.CAM[[1]]))
for(i in 1:length(Results.CAM)){
  CAM <- CAM + Results.CAM[[i]]
}
CAM<-CAM/length(Results.CAM)

In [ ]:
# 3. Perform Consensus Clustering on the Consensus Matrix
####
plotDir = paste0(outDir,p,"/")
res_CCP = ConsensusClusterPlus(as.matrix(CAM),maxK=8,reps=1000,pItem=0.8,pFeature=1,seed = 123,
                               distance="euclidean",clusterAlg="hc", plot=NULL,verbose = FALSE,
                               title = plotDir) 

# Automatically determine the optimal number of clusters from the consensus clustering results

# candidateK<-NULL
# optimalK<-Num_Clusters(res_CCP,  maxK = 8, minClusterNum = 2,outDir = plotDir)
# optimalK<-optimalK[1] # Select the first one if there are multiple optimal clusters
# manually specify the number of clusters
optimalK<-3
Fina_Clus = res_CCP[[optimalK]][["consensusClass"]]

In [ ]:
# 4. Collect clustering results for all patients
# Run Parts 1.1-1.3 for each patient, with p as the patient ID.

Fina_Clus_list[[p]] <- Fina_Clus
seuratObj_list[[p]] <- seuratObj

# Part 2:Select the MPC subpopulation from the clustering results for a single patient

In [ ]:
# 1.Per-cell signature scoring
source("./Function/ModuleExpressionScoring.R")

set.seed(1234)
CellStateScore_List <- lapply(names(Fina_Clus_list), function(p) {
  ModuleExpressionScoring(
    moduleList = signatureList,
    seuratObj = seuratObj_list[[p]]
  )
})
names(CellStateScore_List) <- names(Fina_Clus_list)

In [ ]:
# 2. Select the candidate MPC cluster for each patient
cell_score_list <- lapply(CellStateScore_List, function(scores) {
  rowMeans(do.call(cbind, scores))
})

candidate_cluster_list <- lapply(names(Fina_Clus_list), function(p) {
  cell_score <- cell_score_list[[p]]
  cluster <- Fina_Clus_list[[p]]
  stopifnot(!is.null(names(cluster)),
            setequal(names(cell_score), names(cluster)))
  cluster <- cluster[names(cell_score)]
  names(which.max(tapply(cell_score, cluster, mean)))
})
names(candidate_cluster_list) <- names(Fina_Clus_list)

In [ ]:
# 3. Patient-level size-matched random-cell null (5,000 iterations)
B <- 5000
set.seed(1234)

patient_results <- list()
patient_nulls <- list()

for (p in names(Fina_Clus_list)) {
  cell_score <- cell_score_list[[p]]
  cluster <- Fina_Clus_list[[p]][names(cell_score)]
  candidate_cluster <- candidate_cluster_list[[p]]

  candidate_cells <- names(cluster)[as.character(cluster) == candidate_cluster]
  n_candidate <- length(candidate_cells)
  observed <- mean(cell_score[candidate_cells])

  null_scores <- replicate(
    B, mean(sample(cell_score, size = n_candidate, replace = FALSE))
  )
  null95 <- unname(quantile(null_scores, 0.95))
  p_emp <- (1 + sum(null_scores >= observed)) / (B + 1)

  patient_results[[p]] <- data.frame(
    Patient = p,
    CandidateCluster = candidate_cluster,
    CandidateCells = n_candidate,
    CandidateProportion = n_candidate / length(cell_score),
    ObservedMeanScore = observed,
    Null95 = null95,
    EmpiricalP = p_emp
  )
  patient_nulls[[p]] <- null_scores
}


In [ ]:
# 4. Final patient-level MPC calls
Final_Summary <- dplyr::bind_rows(patient_results)

Final_Summary$FDR <- p.adjust(
  Final_Summary$EmpiricalP, method = "BH"
)

Final_Summary$MPC_detected <- with(
  Final_Summary,
  ObservedMeanScore > Null95 & FDR < 0.05
)

Final_Summary$FinalMPCCluster <- ifelse(
  Final_Summary$MPC_detected,
  Final_Summary$CandidateCluster,
  NA_character_
)

# Additional criterion for downstream biological analyses (>=10 MPC cells)
Final_Summary$Included_downstream <- with(
  Final_Summary,
  MPC_detected & CandidateCells >= 10
)

print(Final_Summary)


In [ ]:
# 5. 上面流程得到的是患者中是否识别到MPC，后面进一步使用10个细胞做阈值筛选下游生物学分析的患者